In [ ]:
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "anndata>=0.10.0",
#     "igraph>=0.10.0",
#     "leidenalg>=0.10.0",
#     "matplotlib>=3.7.0",
#     "numpy>=1.24.0",
#     "pandas>=2.0.0",
#     "scanpy>=1.9.0",
#     "scipy>=1.11.0",
#     "umap-learn>=0.5.0",
# ]
# ///


# PAGA/DPT and scSketch comparison baseline

Purpose: run a clean Scanpy PAGA and diffusion pseudotime baseline on the Monocle 3 C. elegans tutorial AnnData object, then compare scSketch directional selections against matching PAGA/DPT paths.

This notebook uses the AnnData counts and metadata as input, computes PCA/neighbors/Leiden/PAGA, chooses a reproducible root from early embryo-time cells, computes DPT pseudotime, launches scSketch on the PAGA-initialized UMAP, and exports reviewer-facing comparison tables and figures.

## Interpretation note

This is a PAGA/DPT baseline, not a scSketch comparison. PAGA is computed from a high-dimensional neighbor graph built from normalized expression. The UMAP generated here is only a visualization of the Scanpy/PAGA result; it is not used to draw or interpret a scSketch selection in this notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse, stats
from scipy.sparse.csgraph import shortest_path


## Configure inputs

In [ ]:
DATA_FILENAME = "trajectory_task_dataset_v1_participant.h5ad"
DATASET_NAME = Path(DATA_FILENAME).stem

REVISION_DIR_CANDIDATES = [
    Path("."),
    Path("Manuscript/revision_analyses"),
]
REVISION_DIR = next(
    (path for path in REVISION_DIR_CANDIDATES if (path / "data" / DATA_FILENAME).exists()),
    None,
)
if REVISION_DIR is None:
    checked = "; ".join(str(path) for path in REVISION_DIR_CANDIDATES)
    raise FileNotFoundError(f"Could not find {DATA_FILENAME} under: {checked}")

DATA_PATH = REVISION_DIR / "data" / DATA_FILENAME
OUTPUT_DIR = REVISION_DIR / "outputs" / DATASET_NAME / "paga_scsketch_comparison_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COUNT_LAYER = "counts"
N_TOP_GENES = 2000
N_PCS = 50
N_NEIGHBORS = 30
LEIDEN_RESOLUTION = 1.0
LEIDEN_FLAVOR = "igraph"
LEIDEN_N_ITERATIONS = 2
LEIDEN_DIRECTED = False
PAGA_EDGE_THRESHOLD = 0.03
RANDOM_STATE = 0

# Optional PAGA path visualization/export.
RUN_PAGA_PATH_VISUALIZATION = True
PAGA_PATH_CLUSTERS = None  # example: ["0", "5", "11"]; None infers an early-to-late path.
N_PAGA_PATH_GENES = 20
PAGA_PATH_N_AVG = 50
PAGA_PATH_ANNOTATIONS = ["dpt_pseudotime"]

print(f"Using AnnData: {DATA_PATH}")
print(f"Writing outputs to: {OUTPUT_DIR}")


## Load and inspect AnnData

In [ ]:
adata_raw = sc.read_h5ad(DATA_PATH)
print(adata_raw)
print("obs columns:", list(adata_raw.obs.columns))
print("var columns:", list(adata_raw.var.columns))
print("layers:", list(adata_raw.layers.keys()))
print("obsm keys:", list(adata_raw.obsm.keys()))
print("uns keys:", list(adata_raw.uns.keys()))

for col in ["cell.type", "lineage", "time.point", "embryo.time.bin", "embryo.time", "batch"]:
    if col in adata_raw.obs.columns:
        print(f"{col}: {adata_raw.obs[col].nunique()} unique values")


## Preprocess for PAGA

The graph is built from normalized/log-transformed counts, highly variable genes, PCA, and a k-nearest-neighbor graph. The full log-normalized matrix is preserved in `adata.raw` before restricting graph construction to highly variable genes.

In [ ]:
adata = adata_raw.copy()

if COUNT_LAYER in adata.layers:
    adata.X = adata.layers[COUNT_LAYER].copy()
    print(f"Using adata.layers[{COUNT_LAYER!r}] as expression input.")
else:
    print("No counts layer found; using adata.X as expression input.")

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata.copy()

if adata.n_vars > N_TOP_GENES:
    sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES, flavor="seurat")
    adata = adata[:, adata.var["highly_variable"]].copy()
    print(f"Restricted graph construction to {adata.n_vars:,} highly variable genes.")

n_comps = min(N_PCS, adata.n_obs - 1, adata.n_vars - 1)
sc.pp.pca(adata, n_comps=n_comps, random_state=RANDOM_STATE)
sc.pp.neighbors(
    adata,
    n_neighbors=min(N_NEIGHBORS, adata.n_obs - 1),
    n_pcs=n_comps,
    random_state=RANDOM_STATE,
)

print(f"PCA components: {n_comps}")
print("neighbors keys:", adata.uns.get("neighbors", {}).keys())


## Compute Leiden clusters and PAGA

In [ ]:
sc.tl.leiden(
    adata,
    resolution=LEIDEN_RESOLUTION,
    key_added="leiden",
    random_state=RANDOM_STATE,
    flavor=LEIDEN_FLAVOR,
    n_iterations=LEIDEN_N_ITERATIONS,
    directed=LEIDEN_DIRECTED,
)
sc.tl.paga(adata, groups="leiden")

print("Leiden clusters:", adata.obs["leiden"].nunique())
print("Leiden settings:", {
    "resolution": LEIDEN_RESOLUTION,
    "flavor": LEIDEN_FLAVOR,
    "n_iterations": LEIDEN_N_ITERATIONS,
    "directed": LEIDEN_DIRECTED,
    "random_state": RANDOM_STATE,
})
print("PAGA keys:", list(adata.uns["paga"].keys()))


## Summarize clusters and choose a DPT root

The root is chosen reproducibly from the Leiden cluster with the lowest median numeric `embryo.time`. Within that cluster, the root cell is the cell with the smallest numeric `embryo.time`.

In [ ]:
if "embryo.time" not in adata.obs.columns:
    raise KeyError("Expected an 'embryo.time' column for root selection.")

adata.obs["embryo_time_numeric"] = pd.to_numeric(
    adata.obs["embryo.time"].astype(str), errors="coerce"
)

summary_parts = []
cluster_summary = (
    adata.obs.groupby("leiden", observed=True)
    .agg(
        n_cells=("leiden", "size"),
        median_embryo_time=("embryo_time_numeric", "median"),
        min_embryo_time=("embryo_time_numeric", "min"),
        max_embryo_time=("embryo_time_numeric", "max"),
    )
    .reset_index()
)
for col in ["cell.type", "lineage", "time.point", "embryo.time.bin", "batch"]:
    if col in adata.obs.columns:
        mode_by_cluster = (
            adata.obs.groupby("leiden", observed=True)[col]
            .agg(lambda values: values.astype(str).value_counts().index[0])
            .rename(f"top_{col}")
            .reset_index()
        )
        cluster_summary = cluster_summary.merge(mode_by_cluster, on="leiden", how="left")

cluster_summary = cluster_summary.sort_values("median_embryo_time", na_position="last")
cluster_summary.to_csv(OUTPUT_DIR / "leiden_cluster_summary.csv", index=False)

root_cluster = str(cluster_summary.iloc[0]["leiden"])  #pick the Leiden cluster with the earliest median embryo time
root_candidates = np.flatnonzero(                     #finds all cells inside that root cluster that also have finite numeric embryo time
    (adata.obs["leiden"].astype(str).to_numpy() == root_cluster)
    & np.isfinite(adata.obs["embryo_time_numeric"].to_numpy(dtype=float))
)
if len(root_candidates) == 0:
    raise ValueError(f"No finite embryo.time values found in root cluster {root_cluster!r}.")

root_times = adata.obs.iloc[root_candidates]["embryo_time_numeric"].to_numpy(dtype=float)
root_cell_index = int(root_candidates[int(np.argmin(root_times))])     #chooses the cell within the root cluster with the smallest embryo time
adata.uns["iroot"] = root_cell_index    #stores that cell index as the DPT root. Scanpy's sc.tl.dpt(adata) uses adata.uns["iroot"] to decide where pseudotime starts

print(f"Root cluster: {root_cluster}")
print(f"Root cell index: {root_cell_index}")
print(f"Root cell obs name: {adata.obs_names[root_cell_index]}")
print(f"Root embryo.time: {adata.obs.iloc[root_cell_index]['embryo_time_numeric']}")
cluster_summary.head(10)


## Compute DPT pseudotime

In [ ]:
sc.tl.diffmap(adata)     #computes a diffusion map from the neighbor graph. This represents the dataset's manifold structure: cells connected through many short graph paths are considered close in diffusion space.
sc.tl.dpt(adata)         #computes Diffusion Pseudotime using the root cell previously stored in adata.uns["iroot"] = root_cell_index
                         #each cell gets a pseudotime value, usually starting near 0 at the root and increasing as cells move farther away through the graph
valid = np.isfinite(adata.obs["dpt_pseudotime"].to_numpy(dtype=float)) & np.isfinite(      #keeps only cells where both values are valid numbers: 1) known embryo_time_numeric and 2) computed dpt_pseudotime
    adata.obs["embryo_time_numeric"].to_numpy(dtype=float)      
)
spearman = stats.spearmanr(                            #Spearman asks: do cells with later embryo time generally also have higher DPT pseudotime?
    adata.obs.loc[valid, "embryo_time_numeric"],       #Spearman measures monotonic agreement and is rank-based
    adata.obs.loc[valid, "dpt_pseudotime"],
)
pearson = stats.pearsonr(                              #Pearson asks: is the relationship between embryo time and DPT pseudotime approximately linear?
    adata.obs.loc[valid, "embryo_time_numeric"],       #It measures linear correlation
    adata.obs.loc[valid, "dpt_pseudotime"],
)

pseudotime_summary = pd.DataFrame(                                          #creates a one-row summary table with: number of valid cells, Spearman correlation and p-value, Pearson correlation and p-value, root cluster, root cell index, root cell name
    [
        {
            "n_cells_with_finite_embryo_time_and_dpt": int(valid.sum()),    
            "spearman_r": spearman.statistic,
            "spearman_p_value": spearman.pvalue,
            "pearson_r": pearson.statistic,
            "pearson_p_value": pearson.pvalue,
            "root_cluster": root_cluster,
            "root_cell_index": root_cell_index,
            "root_cell_obs_name": str(adata.obs_names[root_cell_index]),
        }
    ]
)
pseudotime_summary.to_csv(OUTPUT_DIR / "pseudotime_validation_summary.csv", index=False)      #writes the validation table to pseudotime_validation_summary.csv
pseudotime_summary


## Plot PAGA and PAGA-initialized UMAP

In [ ]:
sc.pl.paga(                                   #plots the PAGA graph. In that graph: nodes are Leiden clusters, edges are PAGA connectivity relationships between clusters, edges weaker than PAGA_EDGE_THRESHOLD are hidden, nodes are colored by 1)Leiden and 2)embryo_time_numeric
    adata,                                    #The figure helps answer: Do the Leiden/PAGA clusters form a sensible developmental structure, and does embryo time vary coherently across that structure?
    threshold=PAGA_EDGE_THRESHOLD,                      
    color=["leiden", "embryo_time_numeric"],
    show=False,
    frameon=False,
)
plt.savefig(OUTPUT_DIR / "paga_leiden_embryo_time.png", dpi=300, bbox_inches="tight")
plt.show()

sc.tl.umap(adata, init_pos="paga", random_state=RANDOM_STATE)    #computes UMAP embedding. The init_pos="paga" is so that PAGA gives UMAP a cluster-level starting arrangement, so the final UMAP tends to respect the PAGA topology more clearly
                                                                 #this creates adata.obsm["X_umap"] which is later used for plotting and launching scSketch
plot_colors = ["leiden", "dpt_pseudotime", "embryo_time_numeric"]    #sets the default UMAP panels to color by: 1) Leiden clusters, 2)DPT pseudotime, 3)known embryo time. 
for col in ["cell.type", "time.point", "embryo.time.bin"]:       #this loop adds extra metadata panels if those columns exist
    if col in adata.obs.columns:
        plot_colors.append(col)

sc.pl.umap(                      #plots the PAGA-initialized UMAP with all those color panels, then saves it as a file
    adata,
    color=plot_colors,           #Plot the PAGA cluster graph, compute a PAGA-initialized UMAP, and save diagnostic UMAP panels showing whether Leiden clusters, DPT pseudotime, embryo time, and known metadata align sensibly
    show=False,
    frameon=False,
    wspace=0.35,
)
plt.savefig(OUTPUT_DIR / "umap_paga_dpt_diagnostics.png", dpi=300, bbox_inches="tight")  
plt.show()


## Export cell-level PAGA/DPT annotations

In [ ]:
cell_columns = [                #Define required columns.These are the core computed/processed annotations:
    "leiden",                   #leiden: cluster assignment for each cell 
    "dpt_pseudotime",           #DPT pseudotime value for each cell
    "embryo_time_numeric",      #numeric version of the known embryo time metadata 
]
for col in ["cell.type", "lineage", "time.point", "embryo.time.bin", "embryo.time", "batch"]:   #optionally adds extra metadata columns if they exist
    if col in adata.obs.columns:
        cell_columns.append(col)

cell_table = adata.obs[cell_columns].copy()        #creates a DataFrame with only those selected columns
cell_table.insert(0, "obs_name", adata.obs_names.astype(str))     #adds the AnnData cell ID as the first column. This is important because adata.obs_names is the actual cell identifier/index, not a normal column.
cell_table.to_csv(OUTPUT_DIR / "paga_dpt_cell_annotations.csv", index=False)   #writes table to file
cell_table.head()


## Rank genes by association with DPT pseudotime

This gives a baseline PAGA/DPT gene list for later comparison with scSketch directional genes.

In [ ]:
def benjamini_hochberg(p_values):                            #This function applies Benjamini-Hochberg correction to a list of p-values
    p_values = np.asarray(p_values, dtype=float)             #input: one p-value per gene, output: one q-value per gene
    q_values = np.full_like(p_values, np.nan, dtype=float)   #Why this matters: if you test ~20,000 genes, many genes may look significant by chance. BH correction reduces that problem.
    valid = np.isfinite(p_values)
    if not valid.any():
        return q_values
    valid_p = p_values[valid]
    order = np.argsort(valid_p)
    ranked = valid_p[order]
    m = len(ranked)
    adjusted = ranked * m / np.arange(1, m + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    restored = np.empty_like(adjusted)
    restored[order] = adjusted
    q_values[valid] = restored
    return q_values


def pearson_by_gene(X, y):                             #Computes Pearson correlation between each gene and pseudotime
    y = np.asarray(y, dtype=float)                     #input: X: expression matrix, cells x genes. y: DPT pseudotime, one value per cell
    valid = np.isfinite(y)                             #For each gene, it asks: does this gene's expression increase or decrease as DPT pseudotime increases?
    X = X[valid]
    y = y[valid]
    y_centered = y - y.mean()
    y_ss = np.sum(y_centered ** 2)
    n = len(y)

    if sparse.issparse(X):                          
        X = X.tocsr()
        x_sum = np.asarray(X.sum(axis=0)).ravel()
        x_sq_sum = np.asarray(X.power(2).sum(axis=0)).ravel()
        xy_sum = np.asarray(X.T @ y_centered).ravel()
    else:
        X = np.asarray(X, dtype=float)
        x_sum = X.sum(axis=0)
        x_sq_sum = np.sum(X ** 2, axis=0)
        xy_sum = X.T @ y_centered

    x_ss = x_sq_sum - (x_sum ** 2 / n)
    denom = np.sqrt(x_ss * y_ss)
    r = np.divide(xy_sum, denom, out=np.full_like(xy_sum, np.nan, dtype=float), where=denom > 0)  
    df = n - 2
    t_stat = r * np.sqrt(df / np.maximum(1 - r ** 2, 1e-300))
    p_values = 2 * stats.t.sf(np.abs(t_stat), df=df)
    p_values[~np.isfinite(r)] = np.nan
    return r, p_values                               #it returns: r is the Pearson correlation for each gene, p_values is the nominal p-value for that correlation.


raw = adata.raw.to_adata() if adata.raw is not None else adata      #chooses the full normalized/log-transformed expression matrix saved earlier
                                                                    #This is important because earlier the working adata was subset to 2000 highly variable genes for graph construction. But this gene-ranking step wants to test all genes, so it uses adata.raw.
raw = raw[adata.obs_names].copy()                                   #aligns the raw object to the current cell order      
pseudotime = adata.obs["dpt_pseudotime"].to_numpy(dtype=float)      #Extracts the DPT pseudotime values

r, p_values = pearson_by_gene(raw.X, pseudotime)                    #computes correlation between every gene and DPT pseudotime across all cells
gene_ranking = pd.DataFrame(                                        #creates a table with: 
    {
        "gene": raw.var_names.astype(str),                          #gene: internal gene ID, e.g. WormBase ID
        "gene_short_name": raw.var["gene_short_name"].astype(str).to_numpy()    #gene_short_name: readable gene symbol if available
        if "gene_short_name" in raw.var.columns                     
        else raw.var_names.astype(str),
        "dpt_correlation": r,                                       #dpt_correlation: Pearson correlation with DPT pseudotime
        "p_value": p_values,                                        #p_value: nominal p-value
        "q_value": benjamini_hochberg(p_values),                    #q_value: BH-adjusted p-value
    }
)
gene_ranking["abs_correlation"] = gene_ranking["dpt_correlation"].abs()    #adds absolute correlation strength
gene_ranking = gene_ranking.sort_values(["q_value", "abs_correlation"], ascending=[True, False])  #sorts genes by 1)lowest q-value first, 2)strongest absolute correlation second
gene_ranking.to_csv(OUTPUT_DIR / "paga_dpt_gene_ranking_all_cells.csv", index=False)  #writes to a file
gene_ranking.head(25)


## Optional: visualize genes along a PAGA path

This section uses `scanpy.pl.paga_path` to export and visualize expression changes for the top DPT-associated genes along an early-to-late path through the PAGA graph. It is optional because PAGA itself defines topology; this path view is a readable figure/table for selected genes.


In [ ]:
def infer_paga_path_by_embryo_time(adata, cluster_summary, edge_threshold):       #function automatically chooses a PAGA cluster path from early embryo time to late embryo time
    cluster_categories = list(adata.obs["leiden"].cat.categories.astype(str))     #Gets the Leiden cluster names and maps each cluster name to its numeric index in the PAGA connectivity matrix
    cluster_to_idx = {cluster: i for i, cluster in enumerate(cluster_categories)}

    start_cluster = str(cluster_summary.iloc[0]["leiden"])         #chooses cluster with the earliest median embryo time as the start
    end_cluster = str(cluster_summary.dropna(subset=["median_embryo_time"]).iloc[-1]["leiden"])  #chooses cluster with the latest median embryo time as the end

    connectivities = adata.uns["paga"]["connectivities"].copy()  #Gets the PAGA cluster-connectivity graph. This is a matrix where stronger values mean stronger PAGA connection between clusters. 
    if sparse.issparse(connectivities):
        connectivities = connectivities.tocsr()
    else:
        connectivities = sparse.csr_matrix(connectivities)

    thresholded = connectivities.copy()
    thresholded.data[thresholded.data < edge_threshold] = 0      #Removes weak PAGA edges below PAGA_EDGE_THRESHOLD
    thresholded.eliminate_zeros()
    if thresholded.nnz == 0:
        thresholded = connectivities.copy()

    distance_graph = thresholded.copy()
    distance_graph.data = 1.0 / np.maximum(distance_graph.data, 1e-12)   #Converts PAGA connectivity into distance; high connectivity = short distance, low connectivity = long distance... so the shortest path prefers strong PAGA edges
    _, predecessors = shortest_path(distance_graph, directed=False, return_predecessors=True)   #runs graph shortest path

    start_idx = cluster_to_idx[start_cluster]
    end_idx = cluster_to_idx[end_cluster]
    path_indices = [end_idx]
    current = end_idx
    while current != start_idx:                         #then the while loop reconstructs the path from the start cluster to the end cluster. Example of what function returns: ["1", "6", "15", "16"] depending on earliest and latest clusters.
        current = predecessors[start_idx, current]
        if current < 0:
            return [start_cluster, end_cluster]
        path_indices.append(int(current))

    return [cluster_categories[i] for i in reversed(path_indices)]


if RUN_PAGA_PATH_VISUALIZATION:                                 #This conditional runs the visualization
    if PAGA_PATH_CLUSTERS is None:                              #because this is currently True, the notebook runs the visualization
        paga_path_clusters = infer_paga_path_by_embryo_time(    #if you didn't manually specify a path, it automatically infers one from earliest to latest embryo-time cluster
            adata,
            cluster_summary,
            PAGA_EDGE_THRESHOLD,
        )
    else:
        paga_path_clusters = [str(cluster) for cluster in PAGA_PATH_CLUSTERS]    #if you did set PAGA_PATH_CLUSTERS like PAGA_PATH_CLUSTERS = ["0","5","11"] then it would use that manual path instead

    missing_path_clusters = set(paga_path_clusters).difference(adata.obs["leiden"].cat.categories.astype(str))    #checks that every requested path cluster actually exists as a Leiden cluster
    if missing_path_clusters:
        raise ValueError(f"PAGA path clusters are not Leiden categories: {sorted(missing_path_clusters)}")

    paga_path_genes = gene_ranking.head(N_PAGA_PATH_GENES)["gene"].astype(str).tolist()   #Takes the top 20 genes from the all-cell DPT gene ranking
    paga_path_gene_table = gene_ranking.loc[         #creates a small table with: gene ID, gene short name, DPT correlation, q-value, absolute correlation.
        gene_ranking["gene"].astype(str).isin(paga_path_genes),
        ["gene", "gene_short_name", "dpt_correlation", "q_value", "abs_correlation"],
    ].copy()
    gene_label_map = dict(zip(paga_path_gene_table["gene"], paga_path_gene_table["gene_short_name"]))
    paga_path_gene_table.to_csv(OUTPUT_DIR / "paga_path_genes.csv", index=False)    #writes file

    paga_path_table = pd.DataFrame(                    #creates a table listing the path clusters in order, then merges in cluster metadata like cell count and embryo time.
        {"path_position": range(len(paga_path_clusters)), "leiden": paga_path_clusters}
    ).merge(cluster_summary, on="leiden", how="left")
    paga_path_table.to_csv(OUTPUT_DIR / "paga_path_by_embryo_time.csv", index=False)

    plt.close("all")
    paga_path_result = sc.pl.paga_path(        #Scanpy's PAGA path plotting function. It plots expression of selected genes along the chosen PAGA cluster path.
        adata,
        nodes=paga_path_clusters,              #the Leiden clusters to follow
        keys=paga_path_genes,                  #The genes to plot
        use_raw=True,                          #Use adata.raw, meaning the full normalized/log-transformed gene matrix, not only HVGs.
        annotations=PAGA_PATH_ANNOTATIONS,     #plot DPT pseudotime along the path
        groups_key="leiden",                   #the path nodes are Leiden clusters
        n_avg=PAGA_PATH_N_AVG,                 #smooth/average expression over windows of cells for visualization
        normalize_to_zero_one=True,            #Scale each gene's plotted expression from 0 to 1 so expression patterns are visually comparable
        left_margin=0.22,
        ytick_fontsize=8,
        legend_fontsize=7,
        return_data=True,                      #Return the plotted data table, not only the figure
        show=False,
    )
    if isinstance(paga_path_result, tuple):
        _, paga_path_data = paga_path_result
    else:
        paga_path_data = paga_path_result

    fig = plt.gcf()
    fig.set_size_inches(8.5, max(5.0, 0.23 * len(paga_path_genes) + 1.8))
    for axis in fig.axes:
        labels = [label.get_text() for label in axis.get_yticklabels()]  #replaces internal WormBase IDs on the plot/table with readable gene names where possible
        if any(label in gene_label_map for label in labels):
            axis.set_yticklabels([gene_label_map.get(label, label) for label in labels], fontsize=8)

    display_index = [gene_label_map.get(str(index), str(index)) for index in paga_path_data.index]
    paga_path_data_display = paga_path_data.copy()
    paga_path_data_display.index = display_index   #replaces internal WormBase IDs on the plot/table with readable gene names where possible   
    paga_path_data.to_csv(OUTPUT_DIR / "paga_path_expression_top_dpt_genes.csv")
    paga_path_data_display.to_csv(OUTPUT_DIR / "paga_path_expression_top_dpt_genes_display_names.csv")
    plt.savefig(OUTPUT_DIR / "paga_path_top_dpt_genes.png", dpi=300, bbox_inches="tight")
    plt.show()                               #Basically shows this: pick an early-to-late path through the PAGA cluster graph, take the top DPT pseudotime-associated genes, and visualize how those genes change along that PAGA path.
    plt.close(fig)

    print("PAGA path clusters:", paga_path_clusters)
    print(f"Wrote PAGA path outputs to: {OUTPUT_DIR}")
    paga_path_gene_table
else:
    print("Skipping optional PAGA path visualization/export.")


In [ ]:
processed_path = OUTPUT_DIR / "paga_dpt_processed_hvg.h5ad"  #saves the processed AnnData object to disk.
adata.write_h5ad(processed_path)
print(f"Wrote processed AnnData: {processed_path}")

print("Expected outputs:")
for filename in [
    "leiden_cluster_summary.csv",
    "pseudotime_validation_summary.csv",
    "paga_dpt_cell_annotations.csv",
    "paga_dpt_gene_ranking_all_cells.csv",
    "paga_leiden_embryo_time.png",
    "umap_paga_dpt_diagnostics.png",
    "paga_path_by_embryo_time.csv",
    "paga_path_genes.csv",
    "paga_path_expression_top_dpt_genes.csv",
    "paga_path_expression_top_dpt_genes_display_names.csv",
    "paga_path_top_dpt_genes.png",
    "paga_dpt_processed_hvg.h5ad",
]:
    print("-", OUTPUT_DIR / filename)


## Optional: launch scSketch on the PAGA-initialized UMAP

Run this after the PAGA/DPT baseline cells. It uses the Scanpy UMAP already computed with `init_pos="paga"`, so the scSketch selection is made on the same object that contains Leiden clusters, PAGA, and DPT pseudotime.


In [ ]:
from scsketch import ScSketch

if "X_umap" not in adata.obsm:
    raise KeyError(
        "Expected adata.obsm['X_umap'] from the earlier "
        "sc.tl.umap(adata, init_pos='paga', ...) cell. "
        "Run the notebook from the top before launching scSketch."
    )

adata.uns["X_umap_source"] = "Scanpy UMAP initialized from PAGA in paga_scsketch_comparison_baseline.ipynb"

metadata_cols = [
    col for col in [
        "leiden",
        "dpt_pseudotime",
        "embryo_time_numeric",
        "cell.type",
        "lineage",
        "time.point",
        "embryo.time.bin",
        "embryo.time",
        "batch",
    ]
    if col in adata.obs.columns
]

sketch = ScSketch(
    adata=adata,
    metadata_cols=metadata_cols,
    color_by_default="leiden",
    max_genes=0,
    gene_annotation_species="6239",
    reactome_species="6239",
)

print("Launching scSketch on:", adata.uns["X_umap_source"])
print("Available metadata:", metadata_cols)
sketch.show()


## Export the scSketch selection

Run this after drawing, saving, and computing directional genes for the selection you want to compare with PAGA/DPT.


In [ ]:
SCSKETCH_SESSION_PATH = OUTPUT_DIR / "scsketch_on_paga_umap.scsketch.json"
sketch.export_session(SCSKETCH_SESSION_PATH)

sketch.get_action_log().to_csv(
    OUTPUT_DIR / "scsketch_on_paga_umap_action_log.csv",
    index=False,
)

raw_for_gene_names = adata.raw.to_adata() if adata.raw is not None else adata
if "gene_short_name" in raw_for_gene_names.var.columns:
    gene_short_name_lookup = raw_for_gene_names.var["gene_short_name"].astype(str).to_dict()
else:
    gene_short_name_lookup = {str(gene): str(gene) for gene in raw_for_gene_names.var_names}

for selection in sketch.selections.selections:
    safe_name = selection.name.lower().replace(" ", "_").replace("/", "_")
    selected_idx = np.asarray(selection.points, dtype=int)

    selected_cells = adata.obs.iloc[selected_idx].copy()
    selected_cells.insert(0, "selection", selection.name)
    selected_cells.insert(1, "obs_name", adata.obs_names[selected_idx].astype(str))
    selected_cells["UMAP1"] = adata.obsm["X_umap"][selected_idx, 0]
    selected_cells["UMAP2"] = adata.obsm["X_umap"][selected_idx, 1]
    selected_cells.to_csv(
        OUTPUT_DIR / f"scsketch_on_paga_umap_{safe_name}_selected_cells.csv",
        index=False,
    )

    genes = sketch.get_genes(selection.name)
    if not genes.empty:
        genes = genes.copy()
        genes.insert(
            1,
            "gene_short_name",
            genes["gene"].astype(str).map(gene_short_name_lookup).fillna(genes["gene"].astype(str)),
        )
    genes.to_csv(
        OUTPUT_DIR / f"scsketch_on_paga_umap_{safe_name}_directional_genes.csv",
        index=False,
    )

    print(selection.name, "cells:", len(selected_idx), "genes:", len(genes))

print(f"Wrote session: {SCSKETCH_SESSION_PATH}")

## Compare scSketch selections to PAGA/DPT

Run this after exporting the scSketch selections. It maps each saved scSketch path to Leiden/PAGA clusters, infers a connected PAGA cluster path, ranks genes by DPT pseudotime within that path, and writes overlap tables for the reviewer-response comparison.

In [ ]:
import json
import re

MIN_SELECTED_CELLS_PER_CLUSTER = 10
MIN_SELECTION_FRACTION_PER_CLUSTER = 0.02
TOP_N_FOR_OVERLAP = [25, 50, 100, 200]

selection_cell_files = sorted(OUTPUT_DIR.glob("scsketch_on_paga_umap_selection_*_selected_cells.csv"))
selection_gene_files = sorted(OUTPUT_DIR.glob("scsketch_on_paga_umap_selection_*_directional_genes.csv"))

if not selection_cell_files:
    raise FileNotFoundError("No exported scSketch selected-cell CSVs found. Run the export cell above first.")
if not selection_gene_files:
    raise FileNotFoundError("No exported scSketch directional-gene CSVs found. Run the export cell above first.")

print("Selected-cell files:", len(selection_cell_files))
print("Directional-gene files:", len(selection_gene_files))

In [ ]:
def selection_slug(selection_name):                                        #Turns a seletion name into a filename-friendly key.
    return selection_name.lower().replace(" ", "_").replace("/", "_")


def file_slug(path):        #Extracts the selection name from exported filenames; this lets code pair each selected-cell CSV with its matching directional-gene CSV
    match = re.search(r"scsketch_on_paga_umap_(.+)_(?:selected_cells|directional_genes)\.csv$", path.name)
    if not match:
        raise ValueError(f"Could not parse selection name from {path}")
    return match.group(1)


def project_points_to_polyline(points, polyline):      #takes 1)points: selected cell coordinates in UMAP space, 2) polyline: the saved scSketch drawn path/direction
    points = np.asarray(points, dtype=float)           #for each selected cell, it finds the nearest point on the drawn path and assigns that cell a normalized progress value from 0 to 1.
    polyline = np.asarray(polyline, dtype=float)       #cells near start of scSketch -> progress close to 0, cells near end of scSketch path -> progress close to 1
    if len(polyline) < 2:                              #this is later used to order Leiden clusters according to the user-drawn scSketch direction
        return np.full(points.shape[0], np.nan)

    seg_start = polyline[:-1]
    seg_end = polyline[1:]
    seg = seg_end - seg_start
    seg_len = np.linalg.norm(seg, axis=1)
    valid = seg_len > 0
    seg_start = seg_start[valid]
    seg = seg[valid]
    seg_len = seg_len[valid]
    if len(seg_len) == 0:
        return np.full(points.shape[0], np.nan)

    cumulative = np.concatenate([[0.0], np.cumsum(seg_len[:-1])])
    progress = np.empty(points.shape[0], dtype=float)
    for i, point in enumerate(points):
        rel = point - seg_start
        t = np.sum(rel * seg, axis=1) / np.sum(seg * seg, axis=1)
        t = np.clip(t, 0.0, 1.0)
        closest = seg_start + t[:, None] * seg
        dist2 = np.sum((closest - point) ** 2, axis=1)
        j = int(np.argmin(dist2))
        progress[i] = cumulative[j] + t[j] * seg_len[j]
    return progress / np.sum(seg_len)


def reconstruct_path(start_idx, end_idx, predecessors):   #reconstructs the actual sequence of PAGA cluster indices between two clusters after shortest_path(...) has been run.
    if start_idx == end_idx:                              #If the shortest path says: 3->7->19->23 this helps turn the predecessor matrix into that explicit ordered list
        return [start_idx]
    path_indices = [end_idx]
    current = end_idx
    while current != start_idx:
        current = predecessors[start_idx, current]
        if current < 0:
            return []
        path_indices.append(int(current))
    return list(reversed(path_indices))


def connected_paga_path(ordered_clusters, adata, edge_threshold):         #takes Leiden clusters touched by a scSketch selection and forces them into a connected route through the PAGA graph
    cluster_categories = list(adata.obs["leiden"].cat.categories.astype(str))      #reason why its needed: scSketch selection may cover clusters 1, 15, 6, and 16               
    cluster_to_idx = {cluster: i for i, cluster in enumerate(cluster_categories)}   #Ordered by drawing, they might be 1->6->15->16. But PAGA may require intermediate graph connections to make that path valid. This function uses PAGA connectivity to find the strongest connected route between each adjacent pair
    ordered_indices = [cluster_to_idx[c] for c in ordered_clusters if c in cluster_to_idx]   #it does this by: 1)reading adata.uns["paga"]["connectivities"], 2)removing weak edges below edge_threshold, 3)converting connectivity strength into distance via 1/connectivity, 4)running graph shortest path, 5)returning the connected PAGA cluster path as cluster labels
    if len(ordered_indices) <= 1:
        return ordered_clusters
                                                                         
    connectivities = adata.uns["paga"]["connectivities"].copy()
    connectivities = connectivities.tocsr() if sparse.issparse(connectivities) else sparse.csr_matrix(connectivities)

    thresholded = connectivities.copy()
    thresholded.data[thresholded.data < edge_threshold] = 0
    thresholded.eliminate_zeros()
    if thresholded.nnz == 0:
        thresholded = connectivities.copy()

    distance_graph = thresholded.copy()
    distance_graph.data = 1.0 / np.maximum(distance_graph.data, 1e-12)
    _, predecessors = shortest_path(distance_graph, directed=False, return_predecessors=True)

    paga_path_indices = []
    for start, end in zip(ordered_indices[:-1], ordered_indices[1:]):
        segment = reconstruct_path(start, end, predecessors)
        if not segment:
            print(f"No PAGA path found between {cluster_categories[start]} and {cluster_categories[end]}.")
            continue
        if paga_path_indices and segment[0] == paga_path_indices[-1]:
            segment = segment[1:]
        paga_path_indices.extend(segment)

    if not paga_path_indices:
        paga_path_indices = ordered_indices
    return [cluster_categories[i] for i in paga_path_indices]


def rank_genes_for_mask(adata, mask, pseudotime_key="dpt_pseudotime"):  #ranks genes for a subset of cells. Here mask identifies cells belonging to the matched PAGA path. 
    if int(mask.sum()) < 10:                                                  #this function: 1)requires at least 10 cells, 2)uses adata.raw if available, so it can test all genes, not only HVGs, 3)pulls DPT pseudotime for those path cells, 4)computes Pearson correlation between every gene and DPT pseudotime, 5)Computes p-values and BH q-values, 6) sorts by lowest q_value, then strongest abs_correlation.
        raise ValueError("PAGA path has too few cells for gene ranking.")     #the output of the function is a ranked table of PAGA/DPT pseudotime-associated genes for that matched path
    raw = adata.raw.to_adata() if adata.raw is not None else adata
    raw_mask = raw.obs_names.astype(str).isin(adata.obs_names[mask].astype(str))
    raw_path = raw[raw_mask].copy()
    pseudotime = adata.obs.loc[raw_path.obs_names, pseudotime_key].to_numpy(dtype=float)

    r, p_values = pearson_by_gene(raw_path.X, pseudotime)
    ranking = pd.DataFrame(
        {
            "gene": raw_path.var_names.astype(str),
            "gene_short_name": raw_path.var["gene_short_name"].astype(str).to_numpy()
            if "gene_short_name" in raw_path.var.columns
            else raw_path.var_names.astype(str),
            "paga_dpt_correlation": r,
            "p_value": p_values,
            "q_value": benjamini_hochberg(p_values),
        }
    )
    ranking["abs_correlation"] = ranking["paga_dpt_correlation"].abs()
    return ranking.sort_values(["q_value", "abs_correlation"], ascending=[True, False]).reset_index(drop=True)


session_doc = json.loads(SCSKETCH_SESSION_PATH.read_text()) if SCSKETCH_SESSION_PATH.exists() else {}  #reads exported scSketch sesion JSON if it exists
session_by_slug = {                   #builts a lookup from selection slug to the saved selection object. This is important because the session JSON contains the saved scSketch path geometry, which is needed to order cells/clusters along the hand-drawn direction.
    selection_slug(raw_selection.get("name", "")): raw_selection 
    for raw_selection in session_doc.get("selections", [])
}

gene_name_source = adata.raw.to_adata() if adata.raw is not None else adata  #Uses adata.raw when available so readable gene names come from the full gene table.
if "gene_short_name" in gene_name_source.var.columns:                          #then it maps WormBase gene ID -> gene_short_name if gene_short_name exists.
    gene_short_name_lookup = gene_name_source.var["gene_short_name"].astype(str).to_dict()
else:
    gene_short_name_lookup = {str(gene): str(gene) for gene in gene_name_source.var_names}

gene_files_by_slug = {file_slug(path): path for path in selection_gene_files} #creates dictionaries so later code can pair: selection_1 selected cells, selection_1 directional genes
cell_files_by_slug = {file_slug(path): path for path in selection_cell_files}
missing_gene_exports = sorted(set(cell_files_by_slug).difference(gene_files_by_slug))   #checks that every selected-cell file has a matching directional-gene file. If not, it stops with a clear error.
if missing_gene_exports:
    raise FileNotFoundError(f"Missing directional-gene CSVs for: {missing_gene_exports}")

In [ ]:
cluster_summary_rows = []
path_rows = []
overlap_summary_rows = []
shared_gene_rows = []

for slug, cell_path in sorted(cell_files_by_slug.items()):    #for each selection it loads: 1)selected cells from scSketch, 2)directional genes from scSketch. If no directional genes were exported, it skips that selection.
    genes_path = gene_files_by_slug[slug]
    selected_cells = pd.read_csv(cell_path)
    scsketch_genes = pd.read_csv(genes_path)
    if scsketch_genes.empty:
        print(f"Skipping {slug}: no directional genes exported.")
        continue

    selection_name = selected_cells["selection"].iloc[0] if "selection" in selected_cells.columns else slug
    selected_obs_names = set(selected_cells["obs_name"].astype(str))        #this creates a Boolean mask over all cells; True = cell was selected by scSketch, False = cell was not selected
    selected_mask = adata.obs_names.astype(str).isin(selected_obs_names)

    cluster_table = (                                            #summarize selected cells by Leiden Cluster
        adata.obs.assign(scsketch_selected=selected_mask)        #this asks how many cells from each Leiden cluster are included in thsi scSketch selection?
        .groupby("leiden", observed=True)["scsketch_selected"]
        .agg(cluster_size="size", selected_count="sum")
        .reset_index()
    )
    cluster_table["selection"] = selection_name
    cluster_table["selected_fraction_within_cluster"] = cluster_table["selected_count"] / cluster_table["cluster_size"]  #what fraction of this Leiden cluster was selected?
    cluster_table["selected_fraction_of_selection"] = cluster_table["selected_count"] / max(int(selected_mask.sum()), 1)   #what fraction of the scSketch selection came from this Leiden cluster?
    cluster_table = cluster_table.sort_values(    
        ["selected_count", "selected_fraction_within_cluster"],
        ascending=False,
    )

    candidate_clusters = cluster_table.loc[     #choose candidate PAGA clusters; thiskeeps clusters that contribute meaningfully to the scSketch selection
        (cluster_table["selected_count"] >= MIN_SELECTED_CELLS_PER_CLUSTER)
        & (cluster_table["selected_fraction_of_selection"] >= MIN_SELECTION_FRACTION_PER_CLUSTER),
        "leiden",
    ].astype(str).tolist()
    if not candidate_clusters:
        candidate_clusters = (
            cluster_table.loc[cluster_table["selected_count"] > 0, "leiden"]
            .astype(str)
            .head(3)
            .tolist()
        )

    raw_selection = session_by_slug.get(slug, {})      #Order Candidate Clusters By scSketch Drawing Direction
    path = raw_selection.get("path") or []                       #This gets the saved scSketch path geometry from the session JSON.
    ordered_candidate_clusters = candidate_clusters.copy()
    selection_progress = np.full(adata.n_obs, np.nan)         #projects selected cells onto the drawn scSketch path and gives each selected cell a progress value from 0 to 1.

    if "X_umap" in adata.obsm and path:
        selection_progress[selected_mask] = project_points_to_polyline(
            adata.obsm["X_umap"][selected_mask, :2],
            path,
        )
        progress_df = pd.DataFrame(
            {
                "leiden": adata.obs.loc[selected_mask, "leiden"].astype(str).to_numpy(),
                "progress": selection_progress[selected_mask],
            }
        )
        cluster_progress = progress_df.groupby("leiden", observed=True)["progress"].median().dropna().sort_values()   #Then it computes median progress per Leiden cluster: So clusters can be ordered according to the user’s drawn direction.
        ordered_candidate_clusters = [
            cluster for cluster in cluster_progress.index.astype(str) if cluster in candidate_clusters
        ]
        if not ordered_candidate_clusters:
            ordered_candidate_clusters = candidate_clusters.copy()

    paga_path_clusters = connected_paga_path(ordered_candidate_clusters, adata, PAGA_EDGE_THRESHOLD)    #maps the ordered scSketch-touched clusters onto a valid connected route through the PAGA graph
                                                                                                       #this is important because scSketch gives selected cells and drawing geometry, while PAGA gives cluster connectivity. This step creates the matched PAGA path for that scSketch selection.        
    path_mask = adata.obs["leiden"].astype(str).isin(paga_path_clusters).to_numpy(dtype=bool)          #selects all cells whose Leiden cluster lies on that matched PAGA path, then ranks genes by correlation with DPT pseudotime within those path cells.      
    path_gene_ranking = rank_genes_for_mask(adata, path_mask)                                           #this produces PAGA/DPT pseudotime-associated genes for the matched path

    orientation_r = np.nan      #start with "not available" as the default value
    orientation_sign = 1.0
    selected_dpt = adata.obs.loc[selected_mask, "dpt_pseudotime"].to_numpy(dtype=float)
    selected_progress = selection_progress[selected_mask]
    valid_orientation = np.isfinite(selected_progress) & np.isfinite(selected_dpt)
    if valid_orientation.sum() >= 3:     #if enough selected cells have both scSketch progress and DPT pseudotime: orientation_r = correlation between scSketch progress along drawn path and DPT pseudotime among the same selected cells
        orientation_r = float(np.corrcoef(selected_progress[valid_orientation], selected_dpt[valid_orientation])[0, 1])
        if np.isfinite(orientation_r) and orientation_r < 0:
            orientation_sign = -1.0   #if orientation_r is negative, the code flips scSketch correlations for direction comparison. That does not change the gene list overlap. It only helps compare whether genes increase/decrease in the same oriented direction

    scsketch_genes = scsketch_genes.copy()
    if "gene_short_name" not in scsketch_genes.columns:
        scsketch_genes.insert(
            1,
            "gene_short_name",
            scsketch_genes["gene"].astype(str).map(gene_short_name_lookup).fillna(scsketch_genes["gene"].astype(str)),
        )
    scsketch_genes = scsketch_genes.rename(
        columns={"correlation": "scsketch_correlation", "p-value": "scsketch_p_value"}
    )
    scsketch_genes["selection"] = selection_name
    scsketch_genes["selection_dpt_orientation_r"] = orientation_r
    scsketch_genes["orientation_adjusted_scsketch_correlation"] = (
        scsketch_genes["scsketch_correlation"] * orientation_sign
    )

    cluster_table.to_csv(OUTPUT_DIR / f"scsketch_paga_{slug}_cluster_summary.csv", index=False)
    path_gene_ranking.to_csv(OUTPUT_DIR / f"scsketch_paga_{slug}_paga_dpt_gene_ranking.csv", index=False)
    scsketch_genes.to_csv(OUTPUT_DIR / f"scsketch_paga_{slug}_directional_genes_with_names.csv", index=False)

    path_table = pd.DataFrame(
        {
            "selection": selection_name,
            "path_position": range(len(paga_path_clusters)),
            "leiden": paga_path_clusters,
        }
    ).merge(cluster_table, on=["selection", "leiden"], how="left")
    path_table.to_csv(OUTPUT_DIR / f"scsketch_paga_{slug}_path_clusters.csv", index=False)

    merged = scsketch_genes.merge(
        path_gene_ranking,
        on=["gene", "gene_short_name"],
        how="inner",
    )
    merged["same_direction_raw"] = np.sign(merged["scsketch_correlation"]) == np.sign(merged["paga_dpt_correlation"])
    merged["same_direction_orientation_adjusted"] = (
        np.sign(merged["orientation_adjusted_scsketch_correlation"])
        == np.sign(merged["paga_dpt_correlation"])
    )
    merged = merged.sort_values(["q_value", "abs_correlation"], ascending=[True, False]).reset_index(drop=True)
    merged.to_csv(OUTPUT_DIR / f"scsketch_paga_{slug}_gene_overlap.csv", index=False)

    for top_n in TOP_N_FOR_OVERLAP:            
        sc_top = set(scsketch_genes.head(top_n)["gene"].astype(str))
        paga_top = set(path_gene_ranking.head(top_n)["gene"].astype(str))
        overlap = sc_top & paga_top         #this asks: How many of the top scSketch directional genes are also top PAGA/DPT pseudotime-associated genes?
        overlap_summary_rows.append(
            {
                "selection": selection_name,
                "top_n": top_n,
                "scsketch_genes": len(sc_top),
                "paga_dpt_genes": len(paga_top),
                "overlap": len(overlap),
                "jaccard": len(overlap) / len(sc_top | paga_top) if sc_top or paga_top else np.nan,  #computes Jaccard similarity. #ScSketch.get_genes() exports genes sorted by descending correlation, not by absolute correlation or q-value. So the scSketch "top N" here means strongest positive directional correlations in the saved scSketch orientation
                "selection_dpt_orientation_r": orientation_r,
                "candidate_clusters": ",".join(candidate_clusters),
                "paga_path_clusters": ",".join(paga_path_clusters),
            }
        )

    top_shared = merged.loc[      #stores up to 25 genes shared between the top 100 scSketch genes and top 100 PAGA/DPT genes, with both correlation values.
        merged["gene"].isin(set(scsketch_genes.head(100)["gene"]) & set(path_gene_ranking.head(100)["gene"])),
        [
            "selection",
            "gene",
            "gene_short_name",
            "scsketch_correlation",
            "orientation_adjusted_scsketch_correlation",
            "paga_dpt_correlation",
            "same_direction_orientation_adjusted",
            "scsketch_p_value",
            "q_value",
        ],
    ].head(25)
    shared_gene_rows.append(top_shared)

    cluster_summary_rows.append(cluster_table)
    path_rows.append(path_table)
    print(
        f"{selection_name}: selected cells={selected_mask.sum()}, "
        f"candidate clusters={candidate_clusters}, PAGA path={paga_path_clusters}, "
        f"orientation r={orientation_r:.3f}"
    )

all_cluster_summary = pd.concat(cluster_summary_rows, ignore_index=True)
all_paths = pd.concat(path_rows, ignore_index=True)
overlap_summary = pd.DataFrame(overlap_summary_rows)
shared_top_genes = pd.concat(shared_gene_rows, ignore_index=True) if shared_gene_rows else pd.DataFrame()

all_cluster_summary.to_csv(OUTPUT_DIR / "scsketch_paga_all_selection_cluster_summary.csv", index=False)
all_paths.to_csv(OUTPUT_DIR / "scsketch_paga_all_selection_path_clusters.csv", index=False)
overlap_summary.to_csv(OUTPUT_DIR / "scsketch_paga_gene_overlap_summary.csv", index=False)
shared_top_genes.to_csv(OUTPUT_DIR / "scsketch_paga_shared_top100_genes.csv", index=False)

print(f"Wrote comparison outputs to: {OUTPUT_DIR}")
display(overlap_summary)
shared_top_genes.head(30)

## Reviewer-ready summary table and figure

This final section condenses the three scSketch/PAGA comparisons into manuscript-facing outputs. Selection 1 and Selection 2 are marked as the clearest examples because they have the strongest top-gene overlap, while Selection 3 remains in the summary table as an additional sensitivity path.

In [ ]:
SUMMARY_TOP_N = [25, 50, 100, 200]      #sets the top-N overlap cutoffs to report, and labels Selection 1 and Selection 2 as the main examples
PRIMARY_SELECTIONS = {"Selection 1", "Selection 2"}

summary_path = OUTPUT_DIR / "scsketch_paga_gene_overlap_summary.csv"     
shared_path = OUTPUT_DIR / "scsketch_paga_shared_top100_genes.csv"
paths_path = OUTPUT_DIR / "scsketch_paga_all_selection_path_clusters.csv"

for required_path in [summary_path, shared_path, paths_path]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing comparison output: {required_path}")

overlap_summary = pd.read_csv(summary_path)
shared_top_genes = pd.read_csv(shared_path)
all_paths = pd.read_csv(paths_path)

def _file_counts(selection):
    slug = selection.lower().replace(" ", "_").replace("/", "_")
    cells = pd.read_csv(OUTPUT_DIR / f"scsketch_on_paga_umap_{slug}_selected_cells.csv")
    genes = pd.read_csv(OUTPUT_DIR / f"scsketch_on_paga_umap_{slug}_directional_genes.csv")
    return len(cells), len(genes)

summary_rows = []
for selection in sorted(overlap_summary["selection"].unique()):
    rows = overlap_summary.loc[overlap_summary["selection"] == selection].copy()
    selected_cells_n, directional_genes_n = _file_counts(selection)
    path_clusters = (
        all_paths.loc[all_paths["selection"] == selection]
        .sort_values("path_position")["leiden"]
        .astype(str)
        .tolist()
    )
    shared_names = (
        shared_top_genes.loc[shared_top_genes["selection"] == selection, "gene_short_name"]
        .dropna()
        .astype(str)
        .head(10)
        .tolist()
    )
    row = {
        "selection": selection,
        "primary_example": selection in PRIMARY_SELECTIONS,
        "selected_cells": selected_cells_n,
        "scsketch_directional_genes": directional_genes_n,
        "paga_path_clusters": " -> ".join(path_clusters),
        "selection_dpt_orientation_r": rows["selection_dpt_orientation_r"].iloc[0],
        "top_shared_genes": ", ".join(shared_names),
    }
    for top_n in SUMMARY_TOP_N:
        top_row = rows.loc[rows["top_n"] == top_n].iloc[0]
        row[f"top{top_n}_overlap"] = int(top_row["overlap"])
        row[f"top{top_n}_overlap_fraction"] = float(top_row["overlap"] / top_n)
        row[f"top{top_n}_jaccard"] = float(top_row["jaccard"])
    summary_rows.append(row)

revision_summary = pd.DataFrame(summary_rows)
revision_summary.to_csv(OUTPUT_DIR / "scsketch_paga_revision_summary.csv", index=False)

print(f"Wrote summary table: {OUTPUT_DIR / 'scsketch_paga_revision_summary.csv'}")
display(revision_summary)

In [ ]:
plot_overlap = overlap_summary.loc[overlap_summary["top_n"].isin([25, 100])].copy()
plot_overlap["overlap_fraction"] = plot_overlap["overlap"] / plot_overlap["top_n"]
selection_order = ["Selection 1", "Selection 2", "Selection 3"]
selection_order = [selection for selection in selection_order if selection in plot_overlap["selection"].unique()]
colors = {"Selection 1": "#2E6FBB", "Selection 2": "#D68100", "Selection 3": "#6E6E6E"}
markers = {"Selection 1": "o", "Selection 2": "s", "Selection 3": "^"}

fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.4), dpi=200)

bar_width = 0.34
x = np.arange(len(selection_order))
for offset, top_n in [(-bar_width / 2, 25), (bar_width / 2, 100)]:
    values = []
    labels = []
    for selection in selection_order:
        row = plot_overlap.loc[
            (plot_overlap["selection"] == selection) & (plot_overlap["top_n"] == top_n)
        ].iloc[0]
        values.append(row["overlap_fraction"])
        labels.append(f"{int(row['overlap'])}/{top_n}")
    bars = axes[0].bar(
        x + offset,
        values,
        width=bar_width,
        label=f"Top {top_n}",
        color="#4C78A8" if top_n == 25 else "#F58518",
        edgecolor="black",
        linewidth=0.4,
    )
    for bar, label in zip(bars, labels):
        axes[0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.015,
            label,
            ha="center",
            va="bottom",
            fontsize=7,
        )

axes[0].set_xticks(x)
axes[0].set_xticklabels(selection_order, rotation=20, ha="right")
axes[0].set_ylim(0, max(0.55, plot_overlap["overlap_fraction"].max() + 0.1))
axes[0].set_ylabel("Shared genes / top N")
axes[0].set_title("Top-gene overlap")
axes[0].legend(frameon=False, fontsize=8)
axes[0].spines[["top", "right"]].set_visible(False)

for selection in selection_order:
    subset = shared_top_genes.loc[shared_top_genes["selection"] == selection].head(25)
    axes[1].scatter(
        subset["orientation_adjusted_scsketch_correlation"],
        subset["paga_dpt_correlation"],
        label=selection,
        color=colors.get(selection, "#6E6E6E"),
        marker=markers.get(selection, "o"),
        s=28,
        alpha=0.9 if selection in PRIMARY_SELECTIONS else 0.55,
        edgecolor="white",
        linewidth=0.4,
    )

axes[1].plot([0, 1], [0, 1], color="#999999", linewidth=0.8, linestyle="--")
axes[1].set_xlim(0.45, 0.95)
axes[1].set_ylim(0.45, 0.95)
axes[1].set_xlabel("scSketch directional correlation")
axes[1].set_ylabel("PAGA/DPT pseudotime correlation")
axes[1].set_title("Shared top genes")
axes[1].legend(frameon=False, fontsize=8)
axes[1].spines[["top", "right"]].set_visible(False)

fig.suptitle("Agreement between scSketch paths and PAGA/DPT", fontsize=11, y=1.03)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "scsketch_paga_revision_summary_figure.png", dpi=300, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "scsketch_paga_revision_summary_figure.pdf", bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Wrote figure: {OUTPUT_DIR / 'scsketch_paga_revision_summary_figure.png'}")
print(f"Wrote figure: {OUTPUT_DIR / 'scsketch_paga_revision_summary_figure.pdf'}")